# **Perzeptron & neuronale Netze**

Dieses Notebook erklärt die Grundlagen von Perzeptronen und neuronalen Netzen auf einfache Weise. Wir schauen uns an:

1. **Perzeptron** = eine einfache Ja/Nein‑Entscheidung
2. Wie ein Perzeptron **lernt**
3. Warum XOR **nicht** mit einer Geraden geht
4. Wie ein **kleines Netz** das schafft
5. Wie ein Netz **Ziffern** erkennt

In [ ]:
# Bibliotheken laden
import numpy as np
import matplotlib.pyplot as plt

from ipywidgets import interact, FloatSlider, IntSlider, Checkbox

from sklearn.datasets import make_blobs, load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.neural_network import MLPClassifier

import warnings
from sklearn.exceptions import ConvergenceWarning

%matplotlib inline
plt.rcParams["figure.figsize"] = (7, 5)
np.set_printoptions(precision=3, suppress=True)

---

## 1. Was ist ein Perzeptron?

Ein Perzeptron rechnet eine **einfache Summe**:

$$z = w_1 x_1 + w_2 x_2 + b$$

Dann gilt:

- wenn $z \ge 0$ → **1** (feuert)
- wenn $z < 0$ → **0** (feuert nicht)

Die Grenze ist eine **Gerade**.

In [ ]:
def make_2d_dataset(n=80, separation=2.2, seed=0):
    centers = [(-separation, -separation), (separation, separation)]
    X, y01 = make_blobs(n_samples=n, centers=centers, cluster_std=1.0, random_state=seed)
    y = np.where(y01 == 0, 0, 1)
    return X, y

def predict_binary(X, w, b):
    z = X @ w + b
    return np.where(z >= 0, 1, 0)

def plot_decision_2d(X, y, w, b, title=None, show_grid=True):
    fig, ax = plt.subplots()

    if show_grid:
        x_min, x_max = X[:, 0].min() - 1.0, X[:, 0].max() + 1.0
        y_min, y_max = X[:, 1].min() - 1.0, X[:, 1].max() + 1.0
        xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
        grid = np.c_[xx.ravel(), yy.ravel()]
        zz = (grid @ w + b).reshape(xx.shape)
        ax.contourf(xx, yy, zz, levels=[-1e9, 0, 1e9], alpha=0.15, colors=["#ff6b6b", "#4dabf7"])
        ax.contour(xx, yy, zz, levels=[0], colors="k", linewidths=1.5)

    y_pred = predict_binary(X, w, b)
    correct = (y_pred == y)

    ax.scatter(X[~correct & (y == 0), 0], X[~correct & (y == 0), 1], c="#ff6b6b", marker="x", s=60, label="0 falsch")
    ax.scatter(X[~correct & (y == 1), 0], X[~correct & (y == 1), 1], c="#4dabf7", marker="x", s=60, label="1 falsch")

    ax.scatter(X[correct & (y == 0), 0], X[correct & (y == 0), 1], c="#ff6b6b", edgecolor="k", linewidth=0.3, label="0 richtig")
    ax.scatter(X[correct & (y == 1), 0], X[correct & (y == 1), 1], c="#4dabf7", edgecolor="k", linewidth=0.3, label="1 richtig")

    acc = float((y_pred == y).mean())
    ax.set_xlabel("x1")
    ax.set_ylabel("x2")
    ax.grid(True, alpha=0.3)
    ax.set_title(title or f"Entscheidungsgrenze (Genauigkeit: {acc:.2%})")
    ax.legend(loc="best", frameon=True)
    plt.show()

    return acc

### Interaktiv: Schiebe die Regler
Drehe und verschiebe die Gerade, bis möglichst viele Punkte richtig liegen.

In [ ]:
X_demo, y_demo = make_2d_dataset(n=80, separation=2.2, seed=1)

@interact(
    w1=FloatSlider(min=-4.0, max=4.0, step=0.1, value=1.0, description="w1"),
    w2=FloatSlider(min=-4.0, max=4.0, step=0.1, value=1.0, description="w2"),
    b=FloatSlider(min=-4.0, max=4.0, step=0.1, value=0.0, description="b"),
    show_grid=Checkbox(value=True, description="Hintergrund zeigen")
)
def perceptron_by_hand(w1, w2, b, show_grid):
    w = np.array([w1, w2], dtype=float)
    plot_decision_2d(X_demo, y_demo, w, b, title="Perzeptron: von Hand eingestellt", show_grid=show_grid)

---

## 2. Lernen

Das Perzeptron schaut sich Punkte an und **korrigiert sich**, wenn es falsch liegt.

**Epoche** = einmal **alle Punkte** anschauen.

- **Lernrate** = wie gross der Schritt ist.
- **Zufall** = startet mit einer anderen Reihenfolge der Punkte.
- Nach jeder Epoche sieht die Gerade oft schon besser aus.

In [ ]:
def train_perceptron_history(X, y, lr=0.2, epochs=30, shuffle=True, seed=0):
    rng = np.random.default_rng(seed)
    w = np.zeros(X.shape[1], dtype=float)
    b = 0.0
    history = [(w.copy(), float(b))]  # epoch 0

    # mappt 0/1 -> -1/+1 für die einfache Update-Regel
    y_signed = np.where(y == 1, 1, -1)

    for _ in range(epochs):
        idx = np.arange(len(X))
        if shuffle:
            rng.shuffle(idx)
        for i in idx:
            xi = X[i]
            yi = y_signed[i]
            if yi * (np.dot(w, xi) + b) <= 0:
                w = w + lr * yi * xi
                b = b + lr * yi
        history.append((w.copy(), float(b)))

    return history

@interact(
    epoch=IntSlider(min=0, max=30, step=1, value=0, description="Epoche"),
    lr=FloatSlider(min=0.01, max=1.0, step=0.01, value=0.2, description="Lernrate"),
    separation=FloatSlider(min=0.8, max=4.0, step=0.1, value=2.2, description="Trennung"),
    seed=IntSlider(min=0, max=20, step=1, value=1, description="Zufall"),
    shuffle=Checkbox(value=True, description="Mischen")
)
def perceptron_training_view(epoch, lr, separation, seed, shuffle):
    X, y = make_2d_dataset(n=80, separation=separation, seed=seed)
    hist = train_perceptron_history(X, y, lr=lr, epochs=30, shuffle=shuffle, seed=seed)
    w, b = hist[epoch]
    title = f"Training: Epoche {epoch} | lr={lr:.2f} | Trennung={separation:.1f}"
    plot_decision_2d(X, y, w, b, title=title, show_grid=True)
    print("w =", w, "  b =", round(b, 3))

---

## 3. XOR: Eine Gerade reicht nicht

Hier liegen die Punkte so, dass **keine** Gerade alles richtig trennt.
Darum schafft ein einzelnes Perzeptron XOR nicht.

In [ ]:
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y_xor = np.array([0, 1, 1, 0], dtype=int)

def plot_xor_with_line(w, b):
    fig, ax = plt.subplots()
    x_min, x_max = -0.5, 1.5
    y_min, y_max = -0.5, 1.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
    grid = np.c_[xx.ravel(), yy.ravel()]
    zz = (grid @ w + b).reshape(xx.shape)
    ax.contourf(xx, yy, zz, levels=[-1e9, 0, 1e9], alpha=0.15, colors=["#ff6b6b", "#4dabf7"])
    ax.contour(xx, yy, zz, levels=[0], colors="k", linewidths=1.5)

    y_pred = predict_binary(X_xor, w, b)
    for i, (x1, x2) in enumerate(X_xor):
        correct = (y_pred[i] == y_xor[i])
        color = "#4dabf7" if y_xor[i] == 1 else "#ff6b6b"
        marker = "o" if correct else "x"
        if marker == "o":
            ax.scatter([x1], [x2], c=color, s=120, marker=marker, edgecolor="k", linewidth=0.4)
        else:
            ax.scatter([x1], [x2], c=color, s=120, marker=marker, linewidth=0.4)
        ax.text(x1 + 0.05, x2 + 0.05, f"y={y_xor[i]}", fontsize=10)

    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_xlabel("x1")
    ax.set_ylabel("x2")
    ax.set_title(f"XOR: Perzeptron (Genauigkeit: {(y_pred==y_xor).mean():.0%})")
    ax.grid(True, alpha=0.3)
    plt.show()

@interact(
    w1=FloatSlider(min=-6.0, max=6.0, step=0.1, value=1.0, description="w1"),
    w2=FloatSlider(min=-6.0, max=6.0, step=0.1, value=1.0, description="w2"),
    b=FloatSlider(min=-6.0, max=6.0, step=0.1, value=0.0, description="b")
)
def xor_perceptron_by_hand(w1, w2, b):
    w = np.array([w1, w2], dtype=float)
    plot_xor_with_line(w, b)

### Ein kleines Netz kann XOR lösen
Mit einer **Zwischenschicht** kann das Netz eine geknickte Grenze lernen.

### Netzwerk als Bild (mit Beschriftung)
Links sind die **Inputs**, in der Mitte die **Zwischenschicht**, rechts die **Ausgabe**.
Die Verbindungen dazwischen sind die **Gewichte**.

In [ ]:
def draw_network(n_in=2, n_hidden=3, n_out=1):
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.axis("off")

    layers = [n_in, n_hidden, n_out]
    x_positions = [0.1, 0.5, 0.9]

    for layer_idx, (x, n) in enumerate(zip(x_positions, layers)):
        if n == 1:
            y_positions = [0.5]
        else:
            y_positions = np.linspace(0.2, 0.8, n)
        for y in y_positions:
            ax.add_patch(plt.Circle((x, y), 0.03, fill=True, color="#4dabf7"))
        if layer_idx < len(layers) - 1:
            x_next = x_positions[layer_idx + 1]
            n_next = layers[layer_idx + 1]
            y_next = [0.5] if n_next == 1 else np.linspace(0.2, 0.8, n_next)
            for y in y_positions:
                for y2 in y_next:
                    ax.plot([x, x_next], [y, y2], color="#adb5bd", linewidth=1)

    # Beschriftungen
    ax.text(0.1, 0.93, "Inputs", ha="center")
    ax.text(0.5, 0.93, "Zwischenschicht", ha="center")
    ax.text(0.9, 0.93, "Ausgabe", ha="center")
    ax.text(0.3, 0.07, "Gewichte", ha="center", color="#495057")
    ax.text(0.7, 0.07, "Gewichte", ha="center", color="#495057")

    plt.show()

@interact(
    n_in=IntSlider(min=2, max=4, step=1, value=2, description="Inputs"),
    n_hidden=IntSlider(min=1, max=6, step=1, value=3, description="Zwischen"),
    n_out=IntSlider(min=1, max=3, step=1, value=1, description="Ausgabe"),
)
def network_diagram(n_in, n_hidden, n_out):
    draw_network(n_in=n_in, n_hidden=n_hidden, n_out=n_out)

### Wie viele Zwischen‑Neuronen sind sinnvoll?
Wir testen nur einen kleinen Bereich und markieren die **sinnvolle Größe** (gute Genauigkeit ohne zu groß zu werden).

In [ ]:
# Wir testen feste Größen (kleiner Bereich)
max_neurons = 20

digits_local = load_digits()
X_local = digits_local.data.astype(float) / 16.0
y_local = digits_local.target.astype(int)
X_tr, X_te, y_tr, y_te = train_test_split(X_local, y_local, test_size=0.25, random_state=42, stratify=y_local)

sizes = list(range(2, max_neurons + 1))
scores = []
with warnings.catch_warnings():
    warnings.simplefilter("ignore", ConvergenceWarning)
    for h in sizes:
        clf = MLPClassifier(
            hidden_layer_sizes=(h,),
            activation="relu",
            solver="adam",
            alpha=1e-4,
            learning_rate_init=0.01,
            max_iter=120,
            random_state=0,
        )
        clf.fit(X_tr, y_tr)
        scores.append(clf.score(X_te, y_te))

best_idx = int(np.argmax(scores))
best_h = sizes[best_idx]
best_score = scores[best_idx]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(sizes, scores, marker="o", color="#4dabf7")
ax.scatter([best_h], [best_score], color="#f03e3e", s=80, label="Sinnvolle Größe")
ax.set_xlabel("Anzahl Zwischen‑Neuronen")
ax.set_ylabel("Genauigkeit")
ax.set_title(f"Sinnvolle Größe: {best_h} (Genauigkeit {best_score:.2%})")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

In [ ]:
def plot_mlp_xor(hidden_neurons=2, seed=0):
    clf = MLPClassifier(
        hidden_layer_sizes=(hidden_neurons,),
        activation="tanh",
        solver="lbfgs",
        alpha=1e-3,
        max_iter=2000,
        random_state=seed,
    )
    clf.fit(X_xor, y_xor)

    x_min, x_max = -0.5, 1.5
    y_min, y_max = -0.5, 1.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 250), np.linspace(y_min, y_max, 250))
    grid = np.c_[xx.ravel(), yy.ravel()]
    proba = clf.predict_proba(grid)[:, 1].reshape(xx.shape)

    fig, ax = plt.subplots()
    ax.contourf(xx, yy, proba, levels=20, cmap="RdBu", alpha=0.85)
    ax.contour(xx, yy, proba, levels=[0.5], colors="k", linewidths=2)

    for i, (x1, x2) in enumerate(X_xor):
        color = "#4dabf7" if y_xor[i] == 1 else "#ff6b6b"
        ax.scatter([x1], [x2], c=color, s=130, edgecolor="k", linewidth=0.4)
        ax.text(x1 + 0.05, x2 + 0.05, f"y={y_xor[i]}", fontsize=10)

    y_pred = clf.predict(X_xor)
    acc = float((y_pred == y_xor).mean())
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_xlabel("x1")
    ax.set_ylabel("x2")
    ax.set_title(f"MLP löst XOR (Hidden={hidden_neurons}, Genauigkeit={acc:.0%})")
    ax.grid(True, alpha=0.3)
    plt.show()

@interact(
    hidden_neurons=IntSlider(min=1, max=8, step=1, value=2, description="Hidden"),
    seed=IntSlider(min=0, max=10, step=1, value=0, description="Zufall"),
)
def xor_mlp_demo(hidden_neurons, seed):
    plot_mlp_xor(hidden_neurons=hidden_neurons, seed=seed)

---

## 4. Ziffern erkennen (0–9)

Wir nutzen kleine **8×8 Bilder**.
Das Netz lernt: Bild → richtige Zahl.

### So sehen die Ziffern aus
Eine kleine Reihe von Beispiel‑Bildern (8×8 Pixel).

In [ ]:
# Beispielbilder anzeigen
digits_preview = load_digits()
fig, axes = plt.subplots(1, 10, figsize=(10, 2))
for i in range(10):
    axes[i].imshow(digits_preview.images[i], cmap="gray_r", interpolation="nearest")
    axes[i].set_title(str(digits_preview.target[i]))
    axes[i].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
digits = load_digits()
X = digits.data.astype(float) / 16.0  # 0..1
y = digits.target.astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

with warnings.catch_warnings():
    warnings.simplefilter("ignore", ConvergenceWarning)
    mlp = MLPClassifier(
        hidden_layer_sizes=(32,),
        activation="relu",
        solver="adam",
        alpha=1e-4,
        learning_rate_init=0.01,
        max_iter=120,
        random_state=0,
    )
    mlp.fit(X_train, y_train)

y_pred = mlp.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print("MLP Test-Genauigkeit:", f"{acc:.2%}")

### Interaktiv: Bild auswählen
Links das Bild, rechts die Wahrscheinlichkeit für jede Zahl.
Ändere Helligkeit/Cutoff und schau, wann das Netz unsicher wird.

In [ ]:
def softmax(logits):
    logits = logits - np.max(logits)
    exps = np.exp(logits)
    return exps / np.sum(exps)

def relu(z):
    return np.maximum(0.0, z)

W1, W2 = mlp.coefs_
b1, b2 = mlp.intercepts_

def forward_one_hidden(x):
    h = relu(x @ W1 + b1)
    logits = h @ W2 + b2
    p = softmax(logits)
    return h, logits, p

@interact(
    idx=IntSlider(min=0, max=len(X_test) - 1, step=1, value=0, description="Beispiel"),
    gain=FloatSlider(min=0.4, max=1.6, step=0.05, value=1.0, description="Helligkeit"),
    cutoff=FloatSlider(min=0.0, max=0.6, step=0.05, value=0.0, description="Cutoff"),
)
def digits_prediction_demo(idx, gain, cutoff):
    x = X_test[idx].copy()
    x = np.clip(x * gain, 0.0, 1.0)
    x = np.where(x < cutoff, 0.0, x)

    true_label = int(y_test[idx])
    proba = mlp.predict_proba(x.reshape(1, -1))[0]
    pred = int(np.argmax(proba))

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(x.reshape(8, 8), cmap="gray_r", interpolation="nearest")
    axes[0].set_title(f"Bild (Label={true_label})")
    axes[0].axis("off")

    axes[1].bar(np.arange(10), proba, color="#4dabf7")
    axes[1].set_xticks(np.arange(10))
    axes[1].set_ylim(0, 1)
    axes[1].set_title(f"Vorhersage: {pred} (p={proba[pred]:.2f})")
    axes[1].set_xlabel("Ziffer")
    axes[1].set_ylabel("Wahrscheinlichkeit")
    axes[1].grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

### Interaktiv: Was passiert in der Zwischenschicht?

Hier siehst du, welche **Zwischen‑Neuronen** gerade stark aktiv sind.

In [ ]:
@interact(
    idx=IntSlider(min=0, max=len(X_test) - 1, step=1, value=0, description="Beispiel"),
    show_top=IntSlider(min=8, max=32, step=1, value=16, description="Top-N"),
)
def hidden_activations_demo(idx, show_top):
    x = X_test[idx]
    true_label = int(y_test[idx])
    h, logits, p = forward_one_hidden(x)
    top_idx = np.argsort(-h)[:show_top]

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].imshow(x.reshape(8, 8), cmap="gray_r", interpolation="nearest")
    axes[0].set_title(f"Bild (Label={true_label})")
    axes[0].axis("off")

    axes[1].bar(np.arange(show_top), h[top_idx], color="#51cf66")
    axes[1].set_xticks(np.arange(show_top))
    axes[1].set_xticklabels([str(i) for i in top_idx])
    axes[1].set_title("Hidden Layer: stärkste Aktivierungen")
    axes[1].set_xlabel("Neuron-Index (Top-N)")
    axes[1].set_ylabel("Aktivierung")
    axes[1].grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

    pred = int(np.argmax(p))
    print("Forward-Pass (selbst berechnet): Vorhersage =", pred, "| Top-3 =", np.argsort(-p)[:3])

### Interaktiv: Gewichte als Bild

Jedes Zwischen‑Neuron hat 64 Gewichte (ein Gewicht pro Pixel).
Als Bild sieht man grob, **worauf** das Neuron achtet.

In [ ]:
@interact(neuron=IntSlider(min=0, max=W1.shape[1] - 1, step=1, value=0, description="Neuron"))
def show_hidden_neuron_weights(neuron):
    w_img = W1[:, neuron].reshape(8, 8)
    fig, ax = plt.subplots(1, 1, figsize=(4, 4))
    im = ax.imshow(w_img, cmap="coolwarm", interpolation="nearest")
    ax.set_title(f"Hidden-Neuron {neuron}: Gewichte")
    ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.show()

---

## 5. Mini‑Aufgaben

1. Finde eine Gerade, die möglichst viele Punkte trennt.
2. Was passiert bei sehr grosser Lernrate?
3. Zeige, dass XOR mit einer Geraden nicht klappt.
4. Finde ein schwieriges Ziffern‑Bild (mehrere Balken hoch).
5. Verändere Helligkeit/Cutoff: Wann kippt die Vorhersage?